# シグマゼミ 第3回：なぜ「王 − 男 ＋ 女 ≒ 女王」になるのか（言葉のベクトル）
上から順に ▶ を押すだけで動きます。GloVe（40万語×100次元、約130MB）と text8（約30MB）を最初にダウンロードします。

In [ ]:
!git clone -q https://github.com/kakakanokaito/sigmazemi-code.git
%cd sigmazemi-code/ep03_vectors
!wget -q -nc https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz
!wget -q -nc https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/text8.gz

## ① 似ている度合い cos と、王 − 男 ＋ 女
今日の式：cos θ = a・b / (|a| |b|)

In [ ]:
!python3 vectors_demo.py glove-wiki-gigaword-100.gz

## ② なぜ矢印がそろうのか：共起の比（text8 で数える。数十秒かかります）

In [ ]:
!python3 cooccur_ratio.py text8.gz

## ③ いろいろな足し算・引き算（天才 − かしこい ＋ なまけもの など）

In [ ]:
!python3 more_probes.py glove-wiki-gigaword-100.gz

## ④ 自分で試す
下の3語を好きな英単語に変えてみてください（小文字）。

In [ ]:
import gzip, numpy as np
W, V = [], []
with gzip.open('glove-wiki-gigaword-100.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        p = line.rstrip().split(' ')
        if len(p) == 2: continue
        W.append(p[0]); V.append(np.array(p[1:], dtype=np.float32))
V = np.vstack(V); I = {w: i for i, w in enumerate(W)}
U = V / np.linalg.norm(V, axis=1, keepdims=True)

def calc(a, b, c, k=5):
    q = V[I[a]] - V[I[b]] + V[I[c]]
    s = U @ (q / np.linalg.norm(q))
    out = [(W[i], round(float(s[i]), 3)) for i in np.argsort(-s) if W[i] not in (a, b, c)]
    return out[:k]

print(calc('king', 'man', 'woman'))